# Código atualizado


In [1]:
# ============================================
# 0) Imports
# ============================================
import re
import os
import json
import warnings

import numpy as np
import pandas as pd
import numpy as np

from dataclasses import dataclass
from typing import Callable, List, Tuple, Dict, Any, Optional
from datetime import datetime

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_validate
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.impute import SimpleImputer

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report
)

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from itertools import combinations
from sklearn.preprocessing import PolynomialFeatures

import seaborn as sns
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")


In [2]:
# ============================================
# 1) Carregar datasets
# ============================================
DATASET_HC = 'dataset_voz_completo_HC.csv'
DATASET_PD = 'dataset_voz_completo_PD.csv'

df_HC = pd.read_csv(DATASET_HC)
df_PD = pd.read_csv(DATASET_PD)
print("Shape original HC:", df_HC.shape)
print("Shape original DF:", df_PD.shape)

df_HC['status'] = 0
df_PD['status'] = 1

df_concat = pd.concat([df_HC, df_PD], ignore_index=True)

df_concat.to_csv('dataset_concatenado.csv', index=False)

print("Shape final:", df_concat.shape)
print(df_concat)
df_concat.head()


Shape original HC: (41, 88)
Shape original DF: (40, 88)
Shape final: (81, 89)
                                            file_name  group  f0_mean_hz  \
0    AH_064F_7AB034C9-72E4-438B-A9B3-AD7FDA1596C5.wav  HC_AH  131.121658   
1    AH_114S_A89F3548-0B61-4770-B800-2E26AB3908B6.wav  HC_AH  110.595296   
2    AH_121A_BD5BA248-E807-4CB9-8B53-47E7FFE5F8E2.wav  HC_AH  234.182987   
3    AH_123G_559F0706-2238-447C-BA39-DB5933BA619D.wav  HC_AH  102.822765   
4    AH_195B_39DA6A45-F4CC-492A-80D4-FB79049ACC22.wav  HC_AH  111.466057   
..                                                ...    ...         ...   
76  AH_545841223-24FB0419-5BAE-4F9C-8EBC-CD62DA659...  PD_AH  197.627000   
77  AH_545841226-C699FC9E-1E0C-474D-A12A-936DD92B8...  PD_AH  170.152161   
78  AH_545841227-5C77713A-66F1-49D0-BC8A-702C152E6...  PD_AH  258.527907   
79  AH_545847410-D1BA3BB4-1F61-44CA-ACDE-455A8E97E...  PD_AH  125.287455   
80  AH_545880204-EE87D3E2-0D4C-4EAA-ACD7-C3F177AFF...  PD_AH  112.002244   

    f0_st

,file_name,group,f0_mean_hz,f0_std_hz,f0_min_hz,f0_max_hz,f0_cv,f1_mean_hz,f1_std_hz,f2_mean_hz,...,spec_flux_mean,spec_flux_std,spec_energy_low_mean,spec_energy_mid_mean,spec_energy_high_mean,tsallis_sq_amp,shannon_s1_amp,tsallis_sq_f0,shannon_s1_f0,status
0,AH_064F_7AB034C9-72E4-438B-A9B3-AD7FDA1596C5.wav,HC_AH,131.121658,1.382661,127.979975,135.285714,0.010545,670.176352,11.476255,1144.639964,...,0.113668,0.042128,2.135428,12.731040,0.035838,2.175084,3.616453,0.471591,0.543912,0
1,AH_114S_A89F3548-0B61-4770-B800-2E26AB3908B6.wav,HC_AH,110.595296,1.771228,105.192754,115.377726,0.016015,624.156806,50.759496,975.646622,...,0.183278,0.084410,6.280130,4.587857,0.013371,2.150822,3.590799,0.619451,0.727357,0
2,AH_121A_BD5BA248-E807-4CB9-8B53-47E7FFE5F8E2.wav,HC_AH,234.182987,1.199626,231.386563,237.600742,0.005123,475.504882,48.768486,883.969002,...,0.076905,0.025209,34.665394,8.976105,0.020312,2.141979,3.551923,0.000000,-0.000000,0
3,AH_123G_559F0706-2238-447C-BA39-DB5933BA619D.wav,HC_AH,102.822765,0.989926,100.932166,106.161438,0.009628,595.806684,165.885054,1011.652389,...,0.115013,0.034062,16.381947,10.330296,0.079708,2.144335,3.571178,0.000000,-0.000000,0
4,AH_195B_39DA6A45-F4CC-492A-80D4-FB79049ACC22.wav,HC_AH,111.466057,1.052937,109.405717,114.119047,0.009446,698.353948,45.373953,1107.937845,...,0.178199,0.063370,1.263110,4.190388,0.049585,2.157088,3.593633,0.285120,0.352543,0


In [3]:
# ============================================
# 2) Identificar colunas com valores nulos
# ============================================
cols_with_nans = df_concat.columns[df_concat.isnull().any()].tolist()
print(cols_with_nans)

# Configurar e aplicar o Imputador pela Mediana
# A mediana é mais robusta a outliers comuns em sinais de áudio
imputer = SimpleImputer(strategy='median')

# Criamos uma cópia para preservar o dataframe original se necessário
df_imputed = df_concat.copy()

# Aplicamos a transformação apenas nas colunas identificadas
df_imputed[cols_with_nans] = imputer.fit_transform(df_imputed[cols_with_nans])
df_imputed.to_csv('dataset_imputado.csv', index=False)

# 4. Verificação
print(f"Imputação concluída nas colunas: {cols_with_nans}")
print("Total de valores nulos no dataset após o processo:", df_imputed.isnull().sum().sum())

['jitter_local', 'jitter_rap', 'jitter_ppq5', 'tsallis_sq_f0', 'shannon_s1_f0']
Imputação concluída nas colunas: ['jitter_local', 'jitter_rap', 'jitter_ppq5', 'tsallis_sq_f0', 'shannon_s1_f0']
Total de valores nulos no dataset após o processo: 0


In [4]:
# ============================================
# 3) Extração do subject_id
# Padrão: AH_064F_UUID.wav -> Extrai '064F'
# ============================================
def extract_subject_id(name):
    parts = str(name).split('_')
    return parts[1] if len(parts) >= 2 else None

df = pd.read_csv('dataset_imputado.csv')
df["subject_id"] = df["file_name"].apply(extract_subject_id)

df.to_csv('dataset_imputado.csv', index=False)

# checagem rápida
if df["subject_id"].isna().any():
    raise ValueError("Falha ao extrair subject_id de algumas linhas. Verifique o padrão da coluna 'name'.")

print("Nº de sujeitos:", df["subject_id"].nunique())

Nº de sujeitos: 81


In [5]:
# ============================================
# 4) Preparar X, y e grupos
# Removemos metadados e o target das features
# ============================================
cols_to_drop = ["file_name", "group", "status", "subject_id"]
feature_cols = [c for c in df.columns if c not in cols_to_drop]

X = df[feature_cols]
y = df["status"].astype(int)
groups = df["subject_id"]

print("\nBalanceamento (após limpeza):")
print(pd.Series(y).value_counts().rename(index={0: "Controle(0)", 1: "Parkinson(1)"}))


Balanceamento (após limpeza):
status
Controle(0)     41
Parkinson(1)    40
Name: count, dtype: int64


In [ ]:
# ============================================
# 5) Split Treino/Teste SEM leakage (por sujeito) 80/20
#    (apenas cria os índices; não treina nada aqui)
# ============================================
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

# verificando a integridade
train_subjects = set(groups[train_idx])
test_subjects = set(groups[test_idx])
intersection = train_subjects.intersection(test_subjects)

print(f"Sujeitos no Treino: {len(train_subjects)} | Teste: {len(test_subjects)}")
print(f"Leakage check (deve ser 0): {len(intersection)} interseções encontradas.")
if len(intersection) != 0:
    raise ValueError("Houve leakage: mesmos sujeitos em treino e teste.")

In [ ]:
# ============================================
# 6) QuantileClipper
# ============================================
class QuantileClipper(BaseEstimator, TransformerMixin):
    def __init__(self, low=0.01, high=0.99):
        self.low = low
        self.high = high

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        # Calcula os limites (quantis) para cada coluna
        self.lo_ = np.nanquantile(X, self.low, axis=0)
        self.hi_ = np.nanquantile(X, self.high, axis=0)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        # Aplica o limitador: valores abaixo de lo_ viram lo_ 
        # e acima de hi_ viram hi_
        return np.clip(X, self.lo_, self.hi_)

In [ ]:
# ============================================
# 7) analise de multicolinearidade
# ============================================

# Carregar o dataset imputado
df = pd.read_csv('dataset_imputado.csv')

# Remover colunas não numéricas para o cálculo de correlação
cols_to_exclude = ['file_name', 'group', 'status']
df_numeric = df.drop(columns=[col for col in cols_to_exclude if col in df.columns])

# Calcular a matriz de correlação (Pearson)
corr_matrix = df_numeric.corr().abs()

# Selecionar o triângulo superior da matriz
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Identificar variáveis com correlação superior a 0.9
highly_correlated = [column for column in upper.columns if any(upper[column] > 0.9)]

# Preparar dados para visualização (top correlações)
unstacked_corr = upper.unstack().dropna()
sorted_corr = unstacked_corr.sort_values(ascending=False)

# Mostrar as 15 maiores correlações
print("Top 15 correlações mais altas:")
print(sorted_corr.head(15))

# Plotar um Heatmap das correlações
plt.figure(figsize=(16, 12))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', vmin=0, vmax=1)
plt.title("Matriz de Correlação - Diagnóstico de Multicolinearidade")
plt.savefig('correlation_heatmap.png')
plt.show()

# Contagem de colunas problemáticas
print(f"\nNúmero de colunas com correlação > 0.9: {len(highly_correlated)}")

In [ ]:
# ============================================
# 7) analise de multicolinearidade
# ============================================

# 1. Selecionar apenas as colunas de características (X)
cols_to_exclude = ['file_name', 'group', 'status', 'subject_id']
feature_cols = [c for c in df_imputed.columns if c not in cols_to_exclude]
X_temp = df_imputed[feature_cols]

# 2. Cálculo iterativo para remover r > 0.9
corr_matrix = X_temp.corr().abs()
dropped_cols = []
temp_corr = corr_matrix.copy()

while True:
    # Identifica o triângulo superior da matriz de correlação
    upper = temp_corr.where(np.triu(np.ones(temp_corr.shape), k=1).astype(bool))
    
    # Encontra pares que violam o limite de 0.9
    pairs = [(column, row) for column in upper.columns for row in upper.index if upper.loc[row, column] > 0.9]
    
    if not pairs:
        break # Sai do loop quando não houver mais correlações altas
    
    # Lista colunas envolvidas em conflitos
    confliting_cols = list(set([col for pair in pairs for col in pair]))
    
    # Decide remover a coluna com a maior correlação média entre as conflitantes
    mean_corr = temp_corr.loc[confliting_cols, confliting_cols].mean()
    col_to_drop = mean_corr.idxmax()
    
    dropped_cols.append(col_to_drop)
    temp_corr = temp_corr.drop(index=col_to_drop, columns=col_to_drop)

# 3. Aplicar a remoção no dataframe original
df_cleaned = df_imputed.drop(columns=dropped_cols)
df_cleaned.to_csv('dataset_cleaned.csv', index=False)

print(f"Colunas removidas por redundância (r > 0.9): {len(dropped_cols)}")
print(f"Exemplos de colunas descartadas: {dropped_cols[:5]}")

In [ ]:
# ============================================
# 8) Função: consenso de features
# ============================================
def get_consensus_features(rf_imp, xgb_imp, cat_imp, top_k=16):
    """Combine rankings of the 3 models to obtain consensus on the best features"""

    rf_ranked = rf_imp.reset_index(drop=True)
    xgb_ranked = xgb_imp.reset_index(drop=True)
    cat_ranked = cat_imp.reset_index(drop=True)

    rf_rank_dict = {row['feature']: idx + 1 for idx, row in rf_ranked.iterrows()}
    xgb_rank_dict = {row['feature']: idx + 1 for idx, row in xgb_ranked.iterrows()}
    cat_rank_dict = {row['feature']: idx + 1 for idx, row in cat_ranked.iterrows()}

    all_features = set(rf_rank_dict.keys()) | set(xgb_rank_dict.keys()) | set(cat_rank_dict.keys())

    consensus_ranking = {}
    for feature in all_features:
        rf_pos = rf_rank_dict.get(feature, len(rf_ranked) + 1)
        xgb_pos = xgb_rank_dict.get(feature, len(xgb_ranked) + 1)
        cat_pos = cat_rank_dict.get(feature, len(cat_ranked) + 1)
        consensus_ranking[feature] = (rf_pos + xgb_pos + cat_pos) / 3

    sorted_features = sorted(consensus_ranking.items(), key=lambda x: x[1])
    return [feature for feature, _ in sorted_features[:top_k]]


In [ ]:
# Pré-processamento comum
preprocess = ImbPipeline(steps=[
    ("clip", QuantileClipper(0.01, 0.99)),
    ("scale", RobustScaler())
])

X_proc = preprocess.fit_transform(X)


In [ ]:
# Random Forest
rf = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    class_weight="balanced"
)
rf.fit(X_proc, y)
rf_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)


# XGBoost
xgb = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)
xgb.fit(X_proc, y)
xgb_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": xgb.feature_importances_
}).sort_values("importance", ascending=False)


# CatBoost
cat = CatBoostClassifier(
    iterations=500,
    depth=5,
    learning_rate=0.05,
    loss_function="Logloss",
    verbose=False,
    random_seed=42
)
cat.fit(X_proc, y)
cat_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": cat.get_feature_importance()
}).sort_values("importance", ascending=False)

In [ ]:
# Seleciona 16 features principais
top_features = get_consensus_features(
    rf_imp=rf_importance,
    xgb_imp=xgb_importance,
    cat_imp=cat_importance,
    top_k=16
)

print("Top 16 features por consenso:")
for f in top_features:
    print("-", f)

In [ ]:
# ============================================
# 9) Definir as features selecionadas (Top-16)
# ============================================
selected_features = [
    "mfcc11_std",
    "mfcc13_std", 
    "mfcc1_std", 
    "mfcc10_std", 
    "dmfcc3_mean",
    "dmfcc13_std",
    "tsallis_sq_amp",
    "mfcc9_std",
    "shimmer_local",
    "mfcc8_std",
    "mfcc12_std",
    "spec_rolloff_std_hz",
    "mfcc1_mean",
    "dmfcc13_mean",
    "dmfcc7_std",
    "shimmer_apq3",
]

missing = set(selected_features) - set(df.columns)
if missing:
    raise ValueError(f"Colunas selecionadas não encontradas no dataset: {missing}")


In [ ]:
# ============================================
# 10) Criar dataset reduzido (features + status + subject_id)
# ============================================
df_reduced = df[selected_features + ["status", "subject_id"]].copy()
print("\nShape dataset reduzido:", df_reduced.shape)


In [ ]:
# ============================================
# 11) Salvar datasets (limpo e reduzido)
# ============================================
reduced_path = "dataset_reduced_top16.csv"

df_reduced.to_csv(reduced_path, index=False)

print("\nArquivos salvos:")
print("-", reduced_path)

In [ ]:
def make_pipeline(model):
    return ImbPipeline(steps=[
        # 1. Imputação: Garante que não haja nulos entrando nos transformadores
        ("imputer", SimpleImputer(strategy='median')),

        # 2. Clipping: Amortece os outliers extremos antes do escalonamento
        ("clip", QuantileClipper(0.01, 0.99)),
        
        # 3. RobustScaler: Escalonamento robusto baseado em quartis
        ("scale", RobustScaler()),

        # 4. SMOTE: Balanceamento sintético aplicado apenas durante o 'fit' (treino)
        ("smote", SMOTE(random_state=42, k_neighbors=3)),

        # 5. O Classificador (SVC, XGBoost, etc.)
        ("clf", model)
    ])
    

In [ ]:
# =========================================================
# 13) Benchmark: roda GroupKFold CV e devolve tabela resumo
# =========================================================
def benchmark_models(X, y, groups, models_dict, n_splits=5, n_jobs=1):
    cv = GroupKFold(n_splits=n_splits)

    scoring = {
        "bal_acc": "balanced_accuracy",
        "roc_auc": "roc_auc",
        "f1": "f1",
    }

    rows = []
    for name, model in models_dict.items():
        pipe = make_pipeline(model)

        scores = cross_validate(
            pipe, X, y,
            groups=groups,
            cv=cv,
            scoring=scoring,
            n_jobs=n_jobs,
            error_score="raise"
        )

        rows.append({
            "model": name,
            "bal_acc_mean": float(np.mean(scores["test_bal_acc"])),
            "bal_acc_std":  float(np.std(scores["test_bal_acc"])),
            "roc_auc_mean": float(np.mean(scores["test_roc_auc"])),
            "roc_auc_std":  float(np.std(scores["test_roc_auc"])),
            "f1_mean":      float(np.mean(scores["test_f1"])),
            "f1_std":       float(np.std(scores["test_f1"])),
        })

    return pd.DataFrame(rows).sort_values("bal_acc_mean", ascending=False).reset_index(drop=True)


In [ ]:
# =========================================================
# 14) Carregar dataset para benchmarking
#    Escolha UM:
#    A) dataset reduzido Top-16 (recomendado pelo tamanho do dataset)
#    B) dataset completo limpo 
# =========================================================

USE_REDUCED = True  # mude para False se quiser testar com dataset completo

if USE_REDUCED:
    df = pd.read_csv("dataset_reduced_top16.csv")
    feature_cols = [c for c in df.columns if c not in ["status", "subject_id"]]
else:
    df = pd.read_csv("/dataset_cleaned.csv")
    feature_cols = [c for c in df.columns if c not in ["name", "status", "subject_id"]]

X = df[feature_cols].values
y = df["status"].astype(int).values
groups = df["subject_id"].values

print("Dataset usado:", "REDUZIDO Top-16" if USE_REDUCED else "COMPLETO (limpo)")
print("Shape:", df.shape)
print("Nº features:", len(feature_cols))
print("Nº sujeitos:", df["subject_id"].nunique())
print("Balanceamento:", pd.Series(y).value_counts().to_dict())

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
# =========================================================
# 15) Modelos (SEM CatBoost)
# =========================================================
RNG_SEED = 42

models = {
    "logreg_l2": LogisticRegression(
            penalty="l2",
            C=1.0,
            solver="liblinear",
            max_iter=5000,
            class_weight="balanced",
            random_state=RNG_SEED,
            fit_intercept=True,
        ),
    "svc_rbf":  SVC(
            C=100.0,
            gamma=0.1,
            kernel="rbf",
            probability=False,          # necessário para ROC-AUC via predict_proba
            class_weight="balanced",
            random_state=RNG_SEED,      # usado na calibração interna do probability=True
        ),
    # LinearSVC não tem predict_proba -> calibramos para ter probabilidade para ROC-AUC
    "linear_svc_cal": CalibratedClassifierCV(
            estimator=LinearSVC(
                C=1.0,
                class_weight="balanced",
                random_state=RNG_SEED,
                max_iter=10000,
            ),
            method="sigmoid",
            cv=3,
        ),
    # -------------------------
    # Árvores / Ensembles
    # -------------------------
    "random_forest": RandomForestClassifier(
            n_estimators=300,
            max_depth=10,
            min_samples_split=10,
            min_samples_leaf=5,
            max_features="sqrt",
            bootstrap=True,
            class_weight="balanced",
            random_state=RNG_SEED,
            n_jobs=-1,
        ),
    "gradient_boosting": GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=3,
            subsample=1.0,
            random_state=RNG_SEED,
        ),
    "adaboost": AdaBoostClassifier(
            n_estimators=100,
            learning_rate=1.0,            
            random_state=RNG_SEED,
        ),
    "decision_tree": DecisionTreeClassifier(
            criterion="gini",
            max_depth=5,
            min_samples_split=2,
            min_samples_leaf=1,
            random_state=RNG_SEED,
        ),
    # -------------------------
    # KNN (não tem random_state)
    # -------------------------
    "knn": KNeighborsClassifier(
            n_neighbors=7,
            weights="distance",
            metric="minkowski",
            p=2,
        ),
    
    # -------------------------
    # MLP (rede rasa)
    # -------------------------
    "mlp": MLPClassifier(
            hidden_layer_sizes=(50, 50),
            activation="relu",
            solver="adam",
            alpha=0.0001,
            learning_rate="constant",
            learning_rate_init=0.001,
            max_iter=2000,
            tol=1e-5,
            early_stopping=False,
            random_state=RNG_SEED,
        ),
    # -------------------------
    # XGBoost / CatBoost
    # -------------------------
    "xgboost": XGBClassifier(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=1.0,
            reg_lambda=1.0,
            reg_alpha=0.0,
            gamma=0.0,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RNG_SEED,
            n_jobs=-1,
            verbosity=0,
        ),

}


In [ ]:
# =========================================================
# 16) Rodar benchmark (GroupKFold por sujeito)
# =========================================================
results = benchmark_models(X, y, groups, models, n_splits=5, n_jobs=1)

# Mostrar ranking
print("\n=== Ranking por Balanced Accuracy (GroupKFold, sem leakage) ===")
print(results)

In [ ]:
# Modelo final para serialização (pipeline + treino + save)

import numpy as np
import joblib

from sklearn.svm import SVC
from sklearn.preprocessing import RobustScaler
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# sua classe já existente
# from your_module import QuantileClipper


def build_final_svc_pipeline(
    C: float,
    gamma: float,
    rng_seed: int = 42,
    smote_k_neighbors: int = 3,
) -> ImbPipeline:
    """
    Pipeline final (treino e inferência):
        clip -> robustscale -> SMOTE (somente no fit) -> SVC(RBF)

    Observação: imblearn Pipeline aplica SMOTE apenas no fit; no predict o sampler é ignorado.
    """
    clf = SVC(
        C=float(C),
        gamma=float(gamma),
        kernel="rbf",
        probability=False,          # mais rápido; usa decision_function
        class_weight="balanced",
        random_state=rng_seed,
    )

    pipe = ImbPipeline(steps=[
        ("clip", QuantileClipper(0.01, 0.99)),
        ("scale", RobustScaler()),
        ("smote", SMOTE(random_state=rng_seed, k_neighbors=smote_k_neighbors)),
        ("clf", clf),
    ])
    return pipe


def fit_and_serialize_final_model(
    X,
    y,
    C: float,
    gamma: float,
    out_path: str = "final_svc_rbf_pipeline.joblib",
    rng_seed: int = 42,
    smote_k_neighbors: int = 3,
):
    """
    Treina no dataset inteiro (Top-11 ou completo, você decide antes) e salva o pipeline.
    """
    pipe = build_final_svc_pipeline(
        C=C,
        gamma=gamma,
        rng_seed=rng_seed,
        smote_k_neighbors=smote_k_neighbors,
    )
    pipe.fit(X, y)
    joblib.dump(
        {
            "pipeline": pipe,
            "meta": {
                "model": "SVC_RBF",
                "C": float(C),
                "gamma": float(gamma),
                "rng_seed": int(rng_seed),
                "smote_k_neighbors": int(smote_k_neighbors),
                "features_shape": getattr(X, "shape", None),
            },
        },
        out_path,
    )
    return out_path


def fit_and_serialize_final_model_top11(
    X, y,
    C_star: float,
    gamma_star: float,
    t_star: float,
    out_path: str = "final_svc_rbf_top11.joblib",
    rng_seed: int = 42,
    smote_k_neighbors: int = 3,
):
    import joblib
    import numpy as np

    t_star = float(np.clip(t_star, 0.10, 0.90))

    pipe = build_final_svc_pipeline(
        C=C_star,
        gamma=gamma_star,
        rng_seed=rng_seed,
        smote_k_neighbors=smote_k_neighbors,
    )
    pipe.fit(X, y)

    payload = {
        "pipeline": pipe,
        "threshold_t": t_star,
        "meta": {
            "dataset": "dataset_reduced_top11.csv",
            "model": "SVC_RBF",
            "C": float(C_star),
            "gamma": float(gamma_star),
            "threshold_t": float(t_star),
            "rng_seed": int(rng_seed),
            "smote_k_neighbors": int(smote_k_neighbors),
            "n_features": int(X.shape[1]),
            "n_samples": int(X.shape[0]),
        },
    }
    joblib.dump(payload, out_path)
    return out_path
